# 05 — DPO ile Alignment (Hizalama)

**İlan maddesi:** *"RLHF, DPO, PPO veya benzeri model hizalama (alignment) yöntemlerine
aşina olmak."*

## RLHF/PPO vs. DPO
Klasik RLHF üç aşamalıdır: (1) SFT, (2) ayrı bir **ödül modeli** eğitmek, (3) bu ödül
modelini kullanarak **PPO** ile politika modelini güncellemek. Bu, iki model + kararsız
bir RL döngüsü gerektirir.

**DPO (Direct Preference Optimization)**, tercih çiftlerini (chosen/rejected) doğrudan
bir kayıp fonksiyonuna çevirir — ayrı ödül modeli veya RL rollout'u gerekmez. Matematiksel
olarak PPO'nun optimum noktasına eşdeğer bir çözüme ulaşır, ama çok daha kararlı ve
ucuzdur. Bu yüzden küçük ölçekli projelerde tercih ediyoruz.

Bu notebook, 04. notebook'ta ürettiğiniz QLoRA adaptörünün üzerine devam eder.

In [ ]:
# Bu hücre HER notebook'ta ayrı ayrı çalıştırılmalı: Colab'da her sekme/notebook
# genellikle kendi çalışma zamanını (VM) alır, yani /content her seferinde sıfırdanmış
# gibi başlar. Bu hücre kendi kendini onaran bir kurulum yapar:
#   1) Proje klasörü zaten varsa (aynı çalışma zamanında önceki hücre/notebook
#      tarafından kurulmuşsa) hiçbir şey yapmadan devam eder.
#   2) Yoksa Google Drive'ı mount edip, DRIVE_ZIP_PATH'teki zip'i /content'e açar
#      (zip'in içinde 'baykar-nlp-hazirlik/' klasörü kök olarak yer almalı).
#   3) Drive'da zip de yoksa, kendi GitHub reponuzu klonlamanız için bir uyarı basar.
import os, sys

PROJECT_DIR = "/content/baykar-nlp-hazirlik"
DRIVE_ZIP_PATH = "/content/drive/MyDrive/baykar-nlp-hazirlik.zip"

if not os.path.exists(PROJECT_DIR):
    try:
        from google.colab import drive
        # drive.mount() zaten mount edilmişse anında geri döner (idempotent);
        # os.path.exists("/content/drive") ile "mount edilmiş mi" kontrol etmek
        # güvenilmez çünkü klasör, başarısız/yarım bir mount denemesinden sonra
        # bile var olabilir. Bu yüzden koşulsuz çağırıyoruz.
        drive.mount("/content/drive", force_remount=True)
        if os.path.exists(DRIVE_ZIP_PATH):
            import shutil
            shutil.unpack_archive(DRIVE_ZIP_PATH, "/content")
        else:
            print(f"UYARI: {DRIVE_ZIP_PATH} bulunamadı. Zip'i Drive'ınızın köküne "
                  "yükleyin ya da kendi reponuzu klonlayın: "
                  f"!git clone <repo-url> {PROJECT_DIR}")
    except ImportError:
        pass  # Colab dışında (yerelde) çalışıyorsanız bu adım gerekmez.

if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)

sys.path.insert(0, PROJECT_DIR)


## 1. Tercih (preference) veri seti üretimi

'Chosen' = bağlama sadık cevap, 'rejected' = bağlamsız (serbest) üretim.

In [ ]:
from src.alignment.preference_dataset import build_preference_dataset, save_preference_dataset

pairs = build_preference_dataset(max_examples=150)
path = save_preference_dataset(pairs)
print(f"{len(pairs)} tercih çifti kaydedildi -> {path}")
print("\nÖrnek:")
print("Chosen:", pairs[0]["chosen"])
print("Rejected:", pairs[0]["rejected"])


## 2. DPO eğitimi

In [ ]:
from src.alignment.dpo_train import train
from src.config import QLORA_CONFIG

dpo_adapter_path = train(sft_adapter_dir=QLORA_CONFIG.output_dir)
print("DPO adaptörü kaydedildi ->", dpo_adapter_path)


## 3. Karşılaştırma

Aynı soruyu SFT-only ve SFT+DPO modelleriyle karşılaştırın; DPO sonrası cevapların
daha kaynağa sadık ve daha az 'kaçamak' olmasını bekleriz.

In [ ]:
from src.rag.rag_pipeline import answer

q = "Bayraktar Akıncı'nın Bayraktar TB2'den farkı nedir?"
print("SFT-only:\n", answer(q, model_path=QLORA_CONFIG.output_dir)["answer"])
print("\nSFT+DPO:\n", answer(q, model_path=dpo_adapter_path)["answer"])
